# Week 9b: LLM Evaluation — LLM-as-a-Judge
**Applied Generative AI**
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)
*Developed by Prachi Patel, PhD | Open Curriculum — CC BY 4.0*

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Design judge prompts and scoring rubrics** — Clear criteria and few-shot examples
2. **Implement pairwise and pointwise LLM judges** — Compare two outputs or score a single output
3. **Measure judge reliability** — Agreement with humans, inter-judge consistency, position bias
4. **Identify and mitigate judge biases** — Position bias, verbosity bias, self-enhancement
5. **Calibrate LLM judges** — When to trust the judge

---
## ⚙️ Setup — Install & Import

In [ ]:
!pip install -q openai pandas tqdm

### Scoring Criteria Best Practices

- **Operationalize** each criterion (e.g., "completeness" = covers at least 3 of 5 key facts)
- **Avoid overlap** between criteria
- **Include negative examples** in few-shots (what a 1 or 2 looks like)

In [ ]:
import os
import json
import re
from typing import List, Dict, Optional
from getpass import getpass

from openai import OpenAI
import pandas as pd
from tqdm.notebook import tqdm

In [ ]:
# Set OpenAI API key securely (works in Colab)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

---
## 1. What is LLM-as-a-Judge?

### When and Why to Use LLM Judges

- **Scale**: Evaluate thousands of outputs without human annotators
- **Cost**: Cheaper than human evaluation at scale
- **Consistency**: Same rubric applied uniformly
- **Speed**: Real-time or batch evaluation

### Advantages Over Human Eval

| Aspect | Human | LLM Judge |
|--------|-------|----------|
| Scale | Limited | High |
| Cost | High | Lower |
| Consistency | Variable | More uniform |
| Explainability | Often implicit | Can request justification |

### Failure Modes

- **Position bias**: Preferring A over B (or vice versa) based on order
- **Verbosity bias**: Favoring longer outputs
- **Self-enhancement bias**: Preferring outputs similar to the judge's style
- **Rubric drift**: Interpreting criteria differently over time

---
## 2. Designing Judge Prompts

### Rubric Design

A good rubric:
- Defines **clear criteria** (e.g., factuality, completeness, clarity)
- Uses a **consistent scale** (e.g., 1–5)
- Includes **anchor examples** for each score level
- Specifies **output format** (e.g., JSON)

In [ ]:
RUBRIC = """
5 — Perfect: concise, covers facts, no hallucinations.
4 — Minor omissions, mostly correct.
3 — Some inaccuracies or missing key facts.
2 — Major inaccuracies.
1 — Incorrect or irrelevant.
"""

FEWSHOTS = [
    {
        "source": "The moon orbits the Earth every 27.3 days relative to the stars (sidereal month).",
        "A": "The moon orbits Earth every 27.3 days.",
        "B": "The moon completes an orbit around Earth every 24 hours.",
        "winner": "A",
        "score_A": 5,
        "score_B": 1,
        "justification": "B incorrectly claims a 24-hour orbit; A is correct."
    },
]

print("Rubric:", RUBRIC)
print("Few-shot example:", json.dumps(FEWSHOTS[0], indent=2))

---
## 3. Pairwise Comparison

Compare two candidate outputs (A and B) and decide which is better.

In [ ]:
examples = [
    {
        "source": "The Eiffel Tower is located in Paris and was built in 1889 for the World's Fair.",
        "A": "The Eiffel Tower is in Paris and was constructed for the 1889 World's Fair.",
        "B": "The Eiffel Tower is in London and was built in 1888."
    },
    {
        "source": "Water boils at 100°C at sea level.",
        "A": "Water boils at 100 degrees Celsius at sea level.",
        "B": "Water boils at 90°C under normal conditions."
    },
]

pd.DataFrame(examples)

In [ ]:
def make_judge_prompt(
    source: str,
    A: str,
    B: str,
    rubric: str,
    fewshots: List[Dict] = None
) -> str:
    prompt_parts = [
        "System: You are an expert evaluator. Use the rubric and return a JSON object: {winner, score_A, score_B, justification}.",
        f"Rubric:\n{rubric}\n",
    ]
    if fewshots:
        for ex in fewshots:
            prompt_parts.append(
                "Example:\n" + json.dumps({
                    "source": ex["source"], "A": ex["A"], "B": ex["B"],
                    "winner": ex["winner"], "score_A": ex["score_A"], "score_B": ex["score_B"],
                    "justification": ex["justification"]
                }) + "\n"
            )
    prompt_parts.append(f"Source: {source}\nCandidate A: {A}\nCandidate B: {B}\nTask: Which candidate is better? Return only JSON.")
    return "\n".join(prompt_parts)

# Preview prompt for first example
print(make_judge_prompt(examples[0]["source"], examples[0]["A"], examples[0]["B"], RUBRIC, FEWSHOTS)[:500] + "...")

In [ ]:
def judge_with_openai(
    prompt: str,
    model: str = "gpt-4o-mini",
    temperature: float = 0.0,
    max_tokens: int = 200
) -> str:
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a concise JSON-outputting evaluator."},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return completion.choices[0].message.content

def extract_json(text: str) -> dict:
    cleaned = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```")
    try:
        return json.loads(cleaned)
    except Exception:
        pass
    match = re.search(r"\{(?:[^{}]|(?R))*\}", cleaned, flags=re.S)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return {"parse_error": match.group(0)}
    return {"parse_error": text}

In [ ]:
results = []
for ex in tqdm(examples):
    prompt = make_judge_prompt(ex["source"], ex["A"], ex["B"], RUBRIC, FEWSHOTS)
    raw = judge_with_openai(prompt)
    parsed = extract_json(raw)
    results.append({**ex, "raw": raw, "parsed": parsed})

pd.DataFrame(results)

In [ ]:
rows = []
for r in results:
    p = r["parsed"]
    rows.append({
        "source": r["source"][:50] + "...",
        "winner": p.get("winner") if isinstance(p, dict) else None,
        "score_A": p.get("score_A") if isinstance(p, dict) else None,
        "score_B": p.get("score_B") if isinstance(p, dict) else None,
        "justification": p.get("justification", "")[:80] + "..." if isinstance(p, dict) else None
    })

df = pd.DataFrame(rows)
df

---
## 4. Pointwise Scoring

### Inter-Judge Consistency

Run the same evaluation with **multiple judge models** (e.g., gpt-4o-mini vs gpt-4o). High agreement suggests the task is well-defined; low agreement may indicate ambiguous rubrics.

Score a **single output** on multiple dimensions (clarity, correctness, completeness).

In [ ]:
POINTWISE_RUBRIC = """
Score the output on each dimension (1-5):
- clarity: Is it easy to understand?
- correctness: Is it factually accurate?
- completeness: Does it cover key points?

Return JSON: {"clarity": X, "correctness": Y, "completeness": Z}
"""

def pointwise_score(source: str, output: str, rubric: str) -> str:
    prompt = f"Source: {source}\nOutput: {output}\n{rubric}"
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    return r.choices[0].message.content

# Example
out = examples[0]["A"]
src = examples[0]["source"]
score_raw = pointwise_score(src, out, POINTWISE_RUBRIC)
print("Raw response:", score_raw)
print("Parsed:", extract_json(score_raw))

---
## 5. Judge Reliability

### Measuring Agreement with Human Judgments

When you have human labels, compute **accuracy** or **Cohen's kappa**.

In [ ]:
# Example: human labels (ground truth)
human_labels = [
    {"winner": "A", "score_A": 5, "score_B": 1},
    {"winner": "A", "score_A": 5, "score_B": 1},
]

llm_winners = [r.get("winner") for r in df.to_dict("records")]
human_winners = [h["winner"] for h in human_labels]

acc = sum(1 for a, b in zip(llm_winners, human_winners) if a == b) / len(human_winners)
print("Winner agreement accuracy:", acc)

### Detecting Position Bias: A/B Swap Test

Run the same comparison **twice** — once with A first, once with B first. If the judge is unbiased, the winner should be consistent.

In [ ]:
def position_bias_test(
    source: str,
    A: str,
    B: str,
    rubric: str,
    fewshots: List[Dict] = None
) -> Dict:
    """
    Run judge with (A, B) and (B, A). Check if winner flips.
    """
    # Normal order: A, B
    prompt1 = make_judge_prompt(source, A, B, rubric, fewshots)
    raw1 = judge_with_openai(prompt1)
    p1 = extract_json(raw1)
    winner1 = p1.get("winner")

    # Swapped order: B, A (so B is now "A" in prompt, A is "B")
    prompt2 = make_judge_prompt(source, B, A, rubric, fewshots)
    raw2 = judge_with_openai(prompt2)
    p2 = extract_json(raw2)
    # In prompt2, "A" = original B, "B" = original A
    # So winner2="A" means original B won; winner2="B" means original A won
    winner2_map = {"A": "B", "B": "A"}  # map prompt labels to original labels
    winner2_original = winner2_map.get(p2.get("winner"), p2.get("winner"))

    consistent = (winner1 == winner2_original)
    return {
        "winner_A_first": winner1,
        "winner_B_first": winner2_original,
        "consistent": consistent,
        "position_bias_detected": not consistent
    }

# Run on first example
bias_result = position_bias_test(examples[0]["source"], examples[0]["A"], examples[0]["B"], RUBRIC, FEWSHOTS)
print(json.dumps(bias_result, indent=2))

---
## 6. Calibrating LLM Judges

### When to Trust the Judge

- **Calibrate** on a small human-labeled set — measure agreement before scaling
- **Use few-shot examples** from your domain — improves consistency
- **Run position bias tests** — swap A/B and check consistency
- **Compare multiple judges** — use different models or prompts, aggregate
- **Audit edge cases** — manually review disagreements

---
## 7. Exercises

1. **Design a judge for a new domain**: Create a rubric and few-shot examples for evaluating product review summaries (helpfulness, accuracy, conciseness).

2. **Run a position bias test**: Use 5 examples, run A/B and B/A, report how often the winner flips.

3. **Build an end-to-end eval pipeline**: Combine pairwise judge + pointwise scorer + deployment decision logic.

---
## 8. Responsible AI Reflection

**Using AI to evaluate AI**:

- LLM judges can **amplify biases** present in training data — audit for fairness
- **Transparency**: Document your rubric and few-shots; share with stakeholders
- **Human-in-the-loop**: Use LLM judges for scale, but validate with human review on critical cases
- **Avoid circularity**: Don't use the same model to generate and judge — use a different (often stronger) model as judge

---
## Summary & Next Steps

You have learned:
- **Judge prompt design** — Rubrics and few-shot examples
- **Pairwise comparison** — Comparing two outputs with structured JSON
- **Pointwise scoring** — Multi-criteria scoring of single outputs
- **Judge reliability** — Human agreement, position bias detection
- **Calibration** — When to trust the judge

**Next**: Apply these techniques in your own evaluation pipelines. Combine with **Week 9a** metrics for a full eval suite.